# Individual Activity Exploration

This notebook contains individual exploration of the Google Local New York dataset to investigate how activity-related places and user interactions can be defined and filtered.

**Branch:** `ogechika/activity-exploration`

In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter

In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / "Activity-Recommendation-System").exists():
    PROJECT_DIR = PROJECT_DIR / "Activity-Recommendation-System"

DATA_DIR = PROJECT_DIR / "data" / "raw"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

Project directory: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System
Data directory: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw
Data directory exists: True


In [3]:
metadata_files = list(DATA_DIR.glob("meta*.json.gz"))
review_files = list(DATA_DIR.glob("review*.json"))

print("Metadata files:", metadata_files)
print("Review files:", review_files)

Metadata files: [PosixPath('/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw/meta-New_York.json.gz')]
Review files: [PosixPath('/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw/review-New_York_10 2.json')]


In [4]:
if len(metadata_files) != 1:
    raise FileNotFoundError(
        "Expected one metadata file in data/raw/. "
        "Please check the dataset setup instructions."
    )

if len(review_files) != 1:
    raise FileNotFoundError(
        "Expected one review file in data/raw/. "
        "Please check the dataset setup instructions."
    )

metadata_path = metadata_files[0]
review_path = review_files[0]

print("Metadata:", metadata_path.name)
print("Reviews:", review_path.name)

Metadata: meta-New_York.json.gz
Reviews: review-New_York_10 2.json


In [5]:
metadata = pd.read_json(
    metadata_path,
    lines=True,
    compression="gzip"
)

print("Metadata shape:", metadata.shape)
metadata.head()

Metadata shape: (272189, 15)


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url
0,A-Top Insurance,"A-Top Insurance, 1009 Brighton Beach Ave, Broo...",0x89c24469c758686b:0x641f5b84cb9bedfa,NaN,40.578254,-73.959127,"[Insurance broker, Insurance agency]",2.0,4,NaN,"[[Thursday, 10AM–6PM], [Friday, 10AM–6PM], [Sa...",None,Open ⋅ Closes 6PM,"[0x89c24449907718fb:0x31b554a0983f621d, 0x4065...",https://www.google.com/maps/place//data=!4m2!3...
1,T-Mobile,"T-Mobile, 3923 103rd St, Queens, NY 11368",0x89c25fc9494dce47:0x6d63c807b59a55,NaN,40.750146,-73.862536,"[Cell phone store, Electronic parts supplier, ...",3.5,95,$$,"[[Thursday, 10AM–8PM], [Friday, 10AM–8PM], [Sa...","{'Service options': ['In-store shopping', 'Del...",Open ⋅ Closes 8PM,"[0x89c25fc7a91c609f:0xb103d6a261373fd, 0x89c25...",https://www.google.com/maps/place//data=!4m2!3...
2,Ace Hardware,"Ace Hardware, 130 4th Ave, New York, NY 10003",0x89c259992463b6e1:0x4a601a43752541dd,"Chain retailer stocking household tools, suppl...",40.733182,-73.990201,"[Hardware store, Building materials store, Ele...",3.4,8,NaN,"[[Thursday, 8AM–6:30PM], [Friday, 8AM–6:30PM],...",None,Permanently closed,"[0x89c2598340fb6509:0xa996f3db49ba6e78, 0x89c2...",https://www.google.com/maps/place//data=!4m2!3...
3,"Agricultural Consulting Services, Inc.","Agricultural Consulting Services, Inc., 370 Su...",0x89d14bf74320aea5:0x720261cd8960f8cd,NaN,43.054743,-77.649136,[Corporate office],3.0,1,NaN,"[[Thursday, 9AM–5PM], [Friday, 9AM–5PM], [Satu...",None,Open ⋅ Closes 5PM,None,https://www.google.com/maps/place//data=!4m2!3...
4,Long Point,"Long Point, Ledyard, NY 13026",0x89d0bd8d2df8d10f:0x77a8f349f40492fa,NaN,42.715626,-76.710227,[Peninsula],5.0,1,NaN,None,None,NaN,None,"https://www.google.com/maps/place/Ledyard,+NY+..."


In [6]:
# Load an exploratory sample of reviews
reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=100000
)

print("Review sample shape:", reviews_sample.shape)
reviews_sample.head()

Review sample shape: (100000, 8)


,user_id,name,time,rating,text,pics,resp,gmap_id
0,1.137221e+20,Alvin Martinez,1603494795361,5,I'm late to posting this but this store especi...,[{'url': ['https://lh5.googleusercontent.com/p...,None,0x89c25fc9494dce47:0x6d63c807b59a55
1,1.072934e+20,Johnnie Jackson,1620157037403,1,Very dissatisfied I did not get my phone the p...,None,"{'time': 1620268360920, 'text': 'We pride ours...",0x89c25fc9494dce47:0x6d63c807b59a55
2,1.003786e+20,Manie Blazer,1597431662039,5,Excellent very well done with professional car...,None,None,0x89c25fc9494dce47:0x6d63c807b59a55
3,1.149982e+20,Fashion Fiinds,1543773862044,5,Basing my review strictly on the service I rec...,None,"{'time': 1543855317372, 'text': 'Thanks for th...",0x89c25fc9494dce47:0x6d63c807b59a55
4,1.171782e+20,Andres Rieloff,1597279097718,1,Bad! Disorganized. I'm being totally honest. I...,None,None,0x89c25fc9494dce47:0x6d63c807b59a55


In [7]:
category_counts = Counter()

for categories in metadata["category"].dropna():
    if isinstance(categories, list):
        category_counts.update(categories)

category_df = pd.DataFrame(
    category_counts.items(),
    columns=["category", "number_of_places"]
).sort_values(
    "number_of_places",
    ascending=False
).reset_index(drop=True)

print("Number of unique categories:", len(category_df))

category_df.head(50)

Number of unique categories: 3741


,category,number_of_places
0,Restaurant,19316
1,Bar,7925
2,Fast food restaurant,6648
3,Auto repair shop,6334
4,Pizza restaurant,6223
5,Coffee shop,5996
6,Delivery Restaurant,5811
7,Grocery store,5726
8,Beauty salon,5435
9,Takeout Restaurant,5433


## Final Activity Definition (V2)

Uses the team's finalized and validated activity filtering rules.

In [8]:
import re

# broad activity group filter keywords
activity_groups = {
    "Food & Drink": [
        "restaurant",
        "cafe",
        "coffee shop",
        "bakery",
        "bar",
        "brewery",
        "ice cream shop",
        "deli"
    ],

    "Arts & Culture": [
        "museum",
        "art gallery",
        "movie theater",
        "performing arts theater",
        "historical landmark",
        "historical society",
        "cultural center",
        "theater company"
    ],

    "Outdoor & Nature": [
        "park",
        "beach",
        "botanical garden",
        "community garden",
        "nature preserve",
        "hiking area",
        "trail"
    ],

    "Sports & Recreation": [
        "swimming pool",
        "tennis court",
        "golf course",
        "golf club",
        "skating rink",
        "sports complex"
    ],

    "Games & Recreation": [
        "bowling alley",
        "escape room center",
        "arcade",
        "video arcade",
        "amusement park",
        "amusement center",
        "playground",
        "recreation center"
    ],

    "Nightlife & Live Entertainment": [
        "night club",
        "live music venue",
        "music venue",
        "concert hall",
        "comedy club",
        "dance club"
    ],

    "Sightseeing & Attractions": [
        "tourist attraction",
        "scenic spot",
        "observation deck",
        "monument"
    ]
}

# exclusion rules to disqualify places 
exclusions = [
    "gym",
    "fitness center",
    "boxing gym",
    "gymnastics center",
    "rv park",
    "recreational vehicle park",
    "store",
    "shop",
    "shopping mall",
    "supermarket",
    "grocery store",
    "repair",
    "contractor",
    "supplier",
    "wholesaler",
    "dealer",
    "parking",
    "office",
    "coworking",
    "business center",
    "conference center",
    "warehouse",
    "storage",
    "wedding venue",
    "banquet hall"
]

# return true if a keyword appears 
def matches_keyword(text, keyword):
    text = text.lower()
    keyword = keyword.lower()

    pattern = r"\b" + re.escape(keyword) + r"\b"

    return re.search(pattern, text) is not None

# return true if an exclusion term is found
def has_exclusion(categories):

    if not isinstance(categories, list):
        return False

    for category in categories:
        for term in exclusions:
            if matches_keyword(category, term):
                return True

    return False

# returns all broad groups that matched keywords
def get_activity_groups(categories):

    if not isinstance(categories, list):
        return []

    matched_groups = []

    for group, keywords in activity_groups.items():

        for category in categories:

            if any(
                matches_keyword(category, keyword)
                for keyword in keywords
            ):
                matched_groups.append(group)
                break

    return matched_groups

# counts the number of categories of a place that matched a keyword 
def get_confidence(categories):

    if not is_activity(categories):
        return "Not activity"

    match_count = 0

    for category in categories:
        matched = False

        for keywords in activity_groups.values():
            if any(
                matches_keyword(category, keyword)
                for keyword in keywords
            ):
                matched = True
                break

        if matched:
            match_count += 1

    if match_count >= 2:
        return "High"

    elif match_count == 1:
        return "Medium"
 
    else:
        return "Low"

# checks if a place is an activity, returns True/False 
def is_activity(categories):

    if not isinstance(categories, list):
        return False

    if has_exclusion(categories):
        return False


    return len(get_activity_groups(categories)) > 0

In [9]:
metadata["is_activity"] = metadata["category"].apply(is_activity)

activity_metadata = metadata[
    metadata["is_activity"]
].copy()

activity_metadata["activity_groups"] = (
    activity_metadata["category"].apply(get_activity_groups)
)

activity_metadata["confidence"] = (
    activity_metadata["category"].apply(get_confidence)
)

print("Total places:", f"{len(metadata):,}")
print("Final activity places:", f"{len(activity_metadata):,}")
print(
    "Unique activity IDs:",
    f"{activity_metadata['gmap_id'].nunique():,}"
)

Total places: 272,189
Final activity places: 65,448
Unique activity IDs: 65,300


In [10]:
activity_metadata[
    [
        "name",
        "category",
        "activity_groups",
        "confidence",
        "avg_rating",
        "num_of_reviews"
    ]
].sample(20, random_state=42)

,name,category,activity_groups,confidence,avg_rating,num_of_reviews
125518,Premiere Pastry,[Bakery],[Food & Drink],Medium,3.8,8
199785,Village Heros,[Deli],[Food & Drink],Medium,4.7,25
239439,Frame Gourmet Eatery,"[Restaurant, Deli]",[Food & Drink],High,4.2,578
220980,Jesus Taco,"[Tex-Mex restaurant, Delivery Restaurant]",[Food & Drink],High,3.9,137
142255,Byrdcliffe Theatre,[Historical landmark],[Arts & Culture],Medium,4.4,18
256819,La Mercerie,"[French restaurant, Breakfast restaurant, Cafe...",[Food & Drink],High,4.4,558
249245,Morgan's Brooklyn Barbecue,"[Barbecue restaurant, Bar, Restaurant]",[Food & Drink],High,4.0,1303
77548,Buona Fortuna Pizza Inc,"[Pizza restaurant, Italian restaurant, Deliver...",[Food & Drink],High,4.2,98
246765,Mighty Taco,[Mexican restaurant],[Food & Drink],Medium,4.1,937
199454,Gennaro,"[Italian restaurant, Bar, Delivery Restaurant,...",[Food & Drink],High,4.5,325


In [11]:
# Get the gmap_ids for candidate activity places
activity_ids = set(activity_metadata["gmap_id"].dropna().unique())

# Keep reviews associated with candidate activities
activity_reviews_sample = reviews_sample[
    reviews_sample["gmap_id"].isin(activity_ids)
].copy()

print("Reviews in exploratory sample:", len(reviews_sample))
print("Reviews for candidate activities:", len(activity_reviews_sample))

print(
    "Percentage of sample reviews linked to activities:",
    round(len(activity_reviews_sample) / len(reviews_sample) * 100, 2),
    "%"
)

Reviews in exploratory sample: 100000
Reviews for candidate activities: 18494
Percentage of sample reviews linked to activities: 18.49 %


In [12]:
print("Unique users:", activity_reviews_sample["user_id"].nunique())
print("Unique activity places:", activity_reviews_sample["gmap_id"].nunique())
print("Total activity reviews:", len(activity_reviews_sample))

Unique users: 16402
Unique activity places: 892
Total activity reviews: 18494


In [13]:
reviews_per_user = (
    activity_reviews_sample
    .groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

print("Activity reviews per user:")
print(reviews_per_user.describe())

print("\nUsers with at least 2 activity reviews:",
      (reviews_per_user >= 2).sum())

print("Users with at least 3 activity reviews:",
      (reviews_per_user >= 3).sum())

print("Users with at least 5 activity reviews:",
      (reviews_per_user >= 5).sum())

print("Maximum activity reviews by one user:",
      reviews_per_user.max())

Activity reviews per user:
count    16402.000000
mean         1.127545
std          0.386749
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         10.000000
dtype: float64

Users with at least 2 activity reviews: 1866
Users with at least 3 activity reviews: 166
Users with at least 5 activity reviews: 6
Maximum activity reviews by one user: 10


In [14]:
reviews_per_user.value_counts().sort_index().head(10)

1     14536
2      1700
3       118
4        42
5         4
6         1
10        1
Name: count, dtype: int64

## Full Review Interaction Analysis

### Goal

The initial 100,000-review sample showed very sparse user–activity
interactions. However, this sample represents only a small portion of
the full New York review dataset.

This section examines a larger portion of the review data to determine:

1. How many reviews correspond to candidate activity places.
2. How many unique users interact with activity places.
3. How many unique activity places have interactions.
4. How many users have repeated activity interactions.
5. Whether the interaction data is sufficiently dense for recommendation.
6. What filtering criteria may be appropriate when constructing Dataset v1.

In [15]:
print("Review file:", review_path)
print("Candidate activity places:", len(activity_metadata))
print("Candidate activity IDs:", len(activity_ids))

Review file: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw/review-New_York_10 2.json
Candidate activity places: 65448
Candidate activity IDs: 65300


In [16]:
print("Activity metadata rows:", len(activity_metadata))
print("Unique gmap_ids:", activity_metadata["gmap_id"].nunique())
print(
    "Duplicate gmap_id rows:",
    activity_metadata["gmap_id"].duplicated().sum()
)
print(
    "Missing gmap_ids:",
    activity_metadata["gmap_id"].isna().sum()
)

Activity metadata rows: 65448
Unique gmap_ids: 65300
Duplicate gmap_id rows: 148
Missing gmap_ids: 0


In [17]:
duplicate_activity_ids = activity_metadata[
    activity_metadata["gmap_id"].duplicated(keep=False)
].sort_values("gmap_id")

duplicate_activity_ids[
    ["gmap_id", "name", "category"]
].head(20)

,gmap_id,name,category
259,0x4ccab4b3af9c7c85:0x9defa19fe4e295c,Alpine Pizza,[Pizza restaurant]
1728,0x4ccab4b3af9c7c85:0x9defa19fe4e295c,Alpine Pizza,[Pizza restaurant]
1731,0x4ccab7755367e6b3:0xfb5438273acb64bc,North Hudson Beach Park,[Park]
262,0x4ccab7755367e6b3:0xfb5438273acb64bc,North Hudson Beach Park,[Park]
1260,0x4ccaf93f0974c325:0x236fd0ed8e84c49f,Wilmington Recreation Park,[Park]
2729,0x4ccaf93f0974c325:0x236fd0ed8e84c49f,Wilmington Recreation Park,[Park]
1267,0x4ccb056a335054c9:0x2a41677a9e9a9002,Mt. Pisgah Recreation Center,[Park]
2736,0x4ccb056a335054c9:0x2a41677a9e9a9002,Mt. Pisgah Recreation Center,[Park]
1259,0x4ccb6a8c190838a3:0x28a8c623a6875ad,The Adirondack Sky Center & Observatory,"[Observatory, Tourist attraction]"
2728,0x4ccb6a8c190838a3:0x28a8c623a6875ad,The Adirondack Sky Center & Observatory,"[Observatory, Tourist attraction]"


### Duplicate Candidate Activity Records

The finalized V2 activity filter produced 65,448 metadata rows representing
65,300 unique `gmap_id` values. There are 148 duplicate `gmap_id` rows and no
missing IDs.

Because reviews are linked to places using `gmap_id`, the interaction analysis
uses the 65,300 unique activity IDs. Duplicate metadata records therefore do
not cause reviews to be counted multiple times during review filtering.

The duplicate metadata records should be addressed later when constructing
the final cleaned activity dataset.

In [18]:
chunk_size = 250_000

total_reviews_processed = 0
total_activity_reviews = 0
chunk_results = []

for chunk_number, chunk in enumerate(
    pd.read_json(
        review_path,
        lines=True,
        chunksize=chunk_size
    ),
    start=1
):
    activity_chunk = chunk[
        chunk["gmap_id"].isin(activity_ids)
    ].copy()

    chunk_results.append(activity_chunk)

    total_reviews_processed += len(chunk)
    total_activity_reviews += len(activity_chunk)

    print(
        f"Chunk {chunk_number}: "
        f"{total_reviews_processed:,} reviews processed | "
        f"{total_activity_reviews:,} activity reviews found"
    )

Chunk 1: 250,000 reviews processed | 50,310 activity reviews found
Chunk 2: 500,000 reviews processed | 108,103 activity reviews found
Chunk 3: 750,000 reviews processed | 192,731 activity reviews found
Chunk 4: 1,000,000 reviews processed | 294,848 activity reviews found
Chunk 5: 1,250,000 reviews processed | 394,599 activity reviews found
Chunk 6: 1,500,000 reviews processed | 508,498 activity reviews found
Chunk 7: 1,750,000 reviews processed | 626,562 activity reviews found
Chunk 8: 2,000,000 reviews processed | 712,137 activity reviews found
Chunk 9: 2,250,000 reviews processed | 806,866 activity reviews found
Chunk 10: 2,500,000 reviews processed | 905,929 activity reviews found
Chunk 11: 2,750,000 reviews processed | 998,888 activity reviews found
Chunk 12: 3,000,000 reviews processed | 1,096,095 activity reviews found
Chunk 13: 3,250,000 reviews processed | 1,197,626 activity reviews found
Chunk 14: 3,500,000 reviews processed | 1,309,966 activity reviews found
Chunk 15: 3,750,

In [19]:
activity_reviews_full = pd.concat(
    chunk_results,
    ignore_index=True
)

print("Activity review dataset shape:", activity_reviews_full.shape)
print("Total activity reviews:", f"{len(activity_reviews_full):,}")

Activity review dataset shape: (9123126, 8)
Total activity reviews: 9,123,126


In [20]:
print(
    "Unique users:",
    f"{activity_reviews_full['user_id'].nunique():,}"
)

print(
    "Unique reviewed activity places:",
    f"{activity_reviews_full['gmap_id'].nunique():,}"
)

print(
    "Candidate activity places:",
    f"{len(activity_ids):,}"
)

print(
    "Activity review coverage:",
    f"{activity_reviews_full['gmap_id'].nunique() / len(activity_ids) * 100:.2f}%"
)

Unique users: 628,051
Unique reviewed activity places: 53,414
Candidate activity places: 65,300
Activity review coverage: 81.80%


In [21]:
activities_per_user = (
    activity_reviews_full
    .groupby("user_id")["gmap_id"]
    .nunique()
)

print("Unique activity places reviewed per user:")
print(activities_per_user.describe())

print()
print("Users with at least 2 activities:", f"{(activities_per_user >= 2).sum():,}")
print("Users with at least 3 activities:", f"{(activities_per_user >= 3).sum():,}")
print("Users with at least 5 activities:", f"{(activities_per_user >= 5).sum():,}")
print("Users with at least 10 activities:", f"{(activities_per_user >= 10).sum():,}")
print("Users with at least 20 activities:", f"{(activities_per_user >= 20).sum():,}")
print("Maximum activities reviewed by one user:", activities_per_user.max())

Unique activity places reviewed per user:
count    628051.000000
mean         14.342980
std          16.942107
min           1.000000
25%           6.000000
50%           9.000000
75%          16.000000
max        1370.000000
Name: gmap_id, dtype: float64

Users with at least 2 activities: 620,227
Users with at least 3 activities: 604,174
Users with at least 5 activities: 542,001
Users with at least 10 activities: 310,728
Users with at least 20 activities: 121,144
Maximum activities reviewed by one user: 1370


##### Notes
-The initial 100,000-review sample suggested relatively sparse activity histories, with most users appearing only once. Because the sample represents only a small portion of the full New York review dataset, the full review data was processed before drawing conclusions about interaction density.

Using the finalized V2 activity definition, the full dataset produced 9,123,126 activity-related review rows across 628,051 users and 53,414 reviewed activity places.

The median user reviewed 9 unique activity places. 604,174 users have reviewed at least 3 activities, and 542,001 users have reviewed at least 5 activities.

This shows substantially more repeated interaction history than the initial sample suggested and indicates that the dataset contains enough user–activity interactions to investigate personalized recommendation methods.

## Activity-Side Interaction Analysis

Next, we examine how review interactions are distributed across candidate activity places. This helps determine whether interactions are concentrated among a small number of highly popular places or distributed across many activities.

In [22]:
users_per_activity = (
    activity_reviews_full
    .groupby("gmap_id")["user_id"]
    .nunique()
)

print("Unique users per reviewed activity:")
print(users_per_activity.describe())

print()
print("Activities with at least 2 users:", f"{(users_per_activity >= 2).sum():,}")
print("Activities with at least 5 users:", f"{(users_per_activity >= 5).sum():,}")
print("Activities with at least 10 users:", f"{(users_per_activity >= 10).sum():,}")
print("Activities with at least 20 users:", f"{(users_per_activity >= 20).sum():,}")
print("Activities with at least 50 users:", f"{(users_per_activity >= 50).sum():,}")
print("Activities with at least 100 users:", f"{(users_per_activity >= 100).sum():,}")
print("Maximum users for one activity:", users_per_activity.max())

Unique users per reviewed activity:
count    53414.000000
mean       168.647227
std        308.274103
min          5.000000
25%         30.000000
50%         73.000000
75%        184.000000
max       7516.000000
Name: user_id, dtype: float64

Activities with at least 2 users: 53,414
Activities with at least 5 users: 53,414
Activities with at least 10 users: 53,372
Activities with at least 20 users: 45,938
Activities with at least 50 users: 32,750
Activities with at least 100 users: 22,090
Maximum users for one activity: 7516


In [23]:
activity_metadata_unique = (
    activity_metadata
    .drop_duplicates(subset="gmap_id")
    .copy()
)

activity_popularity = (
    users_per_activity
    .rename("unique_users")
    .reset_index()
    .merge(
        activity_metadata_unique[
            ["gmap_id", "name", "category"]
        ],
        on="gmap_id",
        how="left"
    )
    .sort_values("unique_users", ascending=False)
)

activity_popularity.head(20)

,gmap_id,unique_users,name,category
9991,0x89c2599f2db306bb:0x2a5c7f9eea6938a0,7516,Union Square Park,"[Park, City park, Tourist attraction]"
13736,0x89c25b1087680443:0x178bf7c600df7a98,7293,Prospect Park,"[Park, Tourist attraction]"
19986,0x89c2607fc22166fb:0x133d2c68e8326ce1,6974,Flushing Meadows Corona Park,"[Park, Tourist attraction]"
3844,0x89c25854a57fddc7:0xfc2dfa769d275c61,6575,AMC Empire 25,[Movie theater]
5568,0x89c258f55fbea537:0x49d5f074061cac4e,6356,Lincoln Center for the Performing Arts,"[Performing arts theater, Concert hall, Live m..."
27463,0x89c2cfab89e7c29b:0x3f0390ac972b85f1,6223,Bear Mountain State Park,"[State park, Tourist attraction]"
15112,0x89c25bae694479a3:0xb9949385da52e69e,6206,Barclays Center,"[Arena, Sports complex]"
1168,0x89c2443594b7c71d:0x1503bf4bfd6f2c6f,6195,Luna Park in Coney Island,[Amusement park]
10374,0x89c259a6bd917da7:0x79997d3983197d7c,6194,Madison Square Park,"[Park, Tourist attraction]"
1163,0x89c244353f830641:0x7236f2dccccf97c0,6174,Nathan's Famous,"[Hot dog restaurant, Fast food restaurant]"


##### Notes
-Using the finalized V2 activity definition, I processed the full 18.66 million New York reviews rather than relying only on the initial 100,000-review sample.

The final activity set contains 65,300 unique candidate activity IDs. Of these, 53,414 have review interactions in the full dataset, giving approximately 81.8% review coverage.

The resulting interaction data contains 9,123,126 activity review rows from 628,051 users. The median user has reviewed 9 unique activities, and more than 604,000 users have reviewed at least 3.

On the activity side, the median reviewed activity has 73 unique users, and 45,938 activities have at least 20 users.

These results indicate that the dataset contains substantial repeated user and item interaction history and can support further investigation of interaction-based recommendation methods.

In [24]:
total_interactions = len(activity_reviews_full)

top_10 = activity_popularity.head(10)["unique_users"].sum()
top_100 = activity_popularity.head(100)["unique_users"].sum()
top_1000 = activity_popularity.head(1000)["unique_users"].sum()

print("Total review interactions:", f"{total_interactions:,}")
print()
print(
    "Top 10 activities' share:",
    f"{top_10 / total_interactions * 100:.2f}%"
)
print(
    "Top 100 activities' share:",
    f"{top_100 / total_interactions * 100:.2f}%"
)
print(
    "Top 1,000 activities' share:",
    f"{top_1000 / total_interactions * 100:.2f}%"
)

Total review interactions: 9,123,126

Top 10 activities' share: 0.72%
Top 100 activities' share: 4.61%
Top 1,000 activities' share: 19.43%


In [25]:
user_activity_interactions = (
    activity_reviews_full[
        ["user_id", "gmap_id", "rating"]
    ]
    .drop_duplicates(subset=["user_id", "gmap_id"])
    .copy()
)

print(
    "Raw activity review rows:",
    f"{len(activity_reviews_full):,}"
)

print(
    "Unique user-activity interactions:",
    f"{len(user_activity_interactions):,}"
)

print(
    "Repeated user-place reviews removed:",
    f"{len(activity_reviews_full) - len(user_activity_interactions):,}"
)

Raw activity review rows: 9,123,126
Unique user-activity interactions: 9,008,123
Repeated user-place reviews removed: 115,003


##### Notes
-The full dataset contains 9,123,126 activity review rows. After removing repeated reviews of the same activity by the same user, 9,008,123 unique user–activity interactions remain.

This means 115,003 repeated user–place review rows were removed, so deduplication has only a small effect on the overall size of the interaction dataset.

Activity popularity is uneven but distributed across a large number of places. The top 10 activities account for approximately 0.72% of review interactions, the top 100 account for 4.61%, and the top 1,000 account for 19.43%.

This suggests a long-tail popularity distribution that may be relevant when evaluating popularity bias in the recommender system.

In [26]:
print("Rating distribution:")
print(
    user_activity_interactions["rating"]
    .value_counts()
    .sort_index()
)

print()
print("Rating proportions:")
print(
    user_activity_interactions["rating"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print()
print("Average rating:", round(
    user_activity_interactions["rating"].mean(), 2
))

print("Median rating:", user_activity_interactions["rating"].median())

Rating distribution:
rating
1     303060
2     268911
3     883492
4    2262470
5    5290190
Name: count, dtype: int64

Rating proportions:
rating
1     3.36
2     2.99
3     9.81
4    25.12
5    58.73
Name: proportion, dtype: float64

Average rating: 4.33
Median rating: 5.0


In [27]:
user_activity_interactions["rating"].describe()

count    9.008123e+06
mean     4.328559e+00
std      9.997952e-01
min      1.000000e+00
25%      4.000000e+00
50%      5.000000e+00
75%      5.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

##### Notes
-Ratings are available, but the distribution is strongly positive-skewed. Approximately 83.85% of unique user–activity interactions have ratings of 4 or 5 stars. The average rating is 4.33 and the median rating is 5.

This means that although explicit ratings provide information about user preferences, relatively little negative feedback is available compared with positive feedback.

This should be considered when selecting the baseline recommender and evaluation methodology. Ratings could potentially be treated as explicit preference values, or interactions could be treated as implicit/weighted feedback with rating used as an additional preference-strength signal.



## Interaction EDA Summary

- 18.66 million New York reviews were examined.

- The finalized V2 activity definition identified 65,300 unique candidate activity IDs.

- 9,123,126 review rows correspond to candidate activities.

- These reviews represent 628,051 users and 53,414 reviewed activities.

- Approximately 81.8% of candidate activities have at least one review interaction.

- After removing repeated user/place pairs, 9,008,123 unique user–activity interactions remain.

- The median user has reviewed 9 unique activities.

- 604,174 users have reviewed at least 3 activities.

- 542,001 users have reviewed at least 5 activities.

- The median reviewed activity has 73 unique users.

- 45,938 activities have at least 20 unique users.

- Ratings are strongly positive-skewed: approximately 83.85% of unique interactions have ratings of 4 or 5 stars.

- Activity popularity is uneven across the dataset, providing motivation to investigate popularity and long-tail effects during recommendation.

### Main Takeaway

Using the team's finalized V2 activity definition, the New York dataset contains substantially more user–activity interaction history than the initial sample suggested. There appears to be sufficient repeated interaction data to move forward with a personalized baseline recommender.

The next step is to determine appropriate user/item filtering thresholds, establish the train/test methodology, and implement a baseline recommendation approach before incorporating context-aware and popularity-aware components.